Loading the data processed by hdWGCNA in R

In [1]:
import math
import seaborn as sns
import numpy as np
import scanpy as sc
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache

In [2]:
# Selecting the brain region
select_region = "Immune"

In [3]:
# Loading AnnData object
base_path = Path("/data/scRNA/ABCA/AIBS/AWS/expression_matrices/WMB-10Xv3/20230630/")
expr_path = base_path / f"WMB-10Xv3-{select_region}-raw-wmeta.h5ad"
adata = sc.read_h5ad(expr_path)
adata

AnnData object with n_obs × n_vars = 87639 × 32285
    obs: 'cell_barcode', 'barcoded_cell_sample_label', 'library_label', 'feature_matrix_label', 'entity', 'brain_section_label', 'library_method', 'region_of_interest_acronym', 'donor_label', 'donor_genotype', 'donor_sex', 'dataset_label', 'x', 'y', 'cluster_alias', 'neurotransmitter', 'class', 'subclass', 'supertype', 'cluster', 'neurotransmitter_color', 'class_color', 'subclass_color', 'supertype_color', 'cluster_color', 'region_of_interest_order', 'region_of_interest_color'
    var: 'gene_symbol'

In [4]:
# Loading Module Eigengenes dataframe
base_path = Path("/data/scRNA/ABCA/AIBS/AWS/expression_matrices/WMB-10Xv3/20230630/outputs")
modules_df = pd.read_csv(base_path / f"WMB-10Xv3-{select_region}-raw-mc-wgcna-modules.csv", index_col="Unnamed: 0")
modules_df.head()

,gene_name,module,color,kME_Immune-M1,kME_grey,kME_Immune-M2,kME_Immune-M3,kME_Immune-M4,kME_Immune-M5
ENSMUSG00000024610,ENSMUSG00000024610,Immune-M1,yellow,0.559563,-0.026822,0.087691,0.147090,0.058215,0.275142
ENSMUSG00000060586,ENSMUSG00000060586,Immune-M1,yellow,0.869311,0.001982,0.102990,0.249605,0.082369,-0.016836
ENSMUSG00000036594,ENSMUSG00000036594,Immune-M1,yellow,0.865524,0.001619,0.111349,0.246037,0.085992,-0.018585
ENSMUSG00000073421,ENSMUSG00000073421,Immune-M1,yellow,0.813989,-0.001003,0.087103,0.228217,0.082385,0.008726
ENSMUSG00000044162,ENSMUSG00000044162,Immune-M1,yellow,0.766454,0.003662,0.061605,0.206750,0.063295,0.015329


In [5]:
# Loading Module Eigengenes dataframe
base_path = Path("/data/scRNA/ABCA/AIBS/AWS/expression_matrices/WMB-10Xv3/20230630/outputs")
MEs_df = pd.read_csv(base_path / f"WMB-10Xv3-{select_region}-raw-mc-wgcna-MEs.csv", index_col="Unnamed: 0")
MEs_df = MEs_df.drop(columns=["grey"])
MEs_df.head()

,blue,turquoise,green,yellow,brown
TGTGCGGAGTCCTGTA-216_A01,-8.505898,30.283247,0.117752,-0.792412,-1.235391
AAGAACAGTGGTGATG-216_D01,-8.551579,30.065327,-0.688576,-0.734662,-0.581734
ACAGAAATCGACGTCG-471_B04,-7.861430,22.244992,-0.688576,-0.228925,-0.276623
TTGCCTGCAGGAGGAG-216_C01,-9.094108,36.717444,-0.688576,-0.736562,-1.290367
TCCATCGTCTCGCTTG-594_B02,-7.538399,32.596541,0.321969,2.167889,1.802641


In [6]:
# Create a mapping from color to module
color_to_module = modules_df.set_index("color")["module"].to_dict()

# Rename the columns of MEs_df
MEs_df.rename(columns=color_to_module, inplace=True)

# Sort the columns of MEs_df alphabetically
MEs_df = MEs_df.reindex(sorted(MEs_df.columns), axis=1)
MEs_df.head()

,Immune-M1,Immune-M2,Immune-M3,Immune-M4,Immune-M5
TGTGCGGAGTCCTGTA-216_A01,-0.792412,30.283247,-1.235391,0.117752,-8.505898
AAGAACAGTGGTGATG-216_D01,-0.734662,30.065327,-0.581734,-0.688576,-8.551579
ACAGAAATCGACGTCG-471_B04,-0.228925,22.244992,-0.276623,-0.688576,-7.861430
TTGCCTGCAGGAGGAG-216_C01,-0.736562,36.717444,-1.290367,-0.688576,-9.094108
TCCATCGTCTCGCTTG-594_B02,2.167889,32.596541,1.802641,0.321969,-7.538399


In [7]:
# Getting the list of modules
modules = MEs_df.columns.tolist()

In [8]:
# Adding MEs to AnnData object
merged_df = pd.merge(adata.obs, MEs_df, left_index=True, right_index=True)
merged_df.head()

,cell_barcode,barcoded_cell_sample_label,library_label,feature_matrix_label,entity,brain_section_label,library_method,region_of_interest_acronym,donor_label,donor_genotype,...,subclass_color,supertype_color,cluster_color,region_of_interest_order,region_of_interest_color,Immune-M1,Immune-M2,Immune-M3,Immune-M4,Immune-M5
TGTGCGGAGTCCTGTA-216_A01,TGTGCGGAGTCCTGTA,216_A01,L8TX_200206_01_F03,WMB-10Xv3-CB,cell,NaN,10Xv3,CB,Slc32a1-IRES-Cre;Ai14-507773,Slc32a1-IRES-Cre/wt;Ai14(RCL-tdT)/wt,...,#66493D,#32662E,#EEFF99,28,#CC0026,-0.792412,30.283247,-1.235391,0.117752,-8.505898
AAGAACAGTGGTGATG-216_D01,AAGAACAGTGGTGATG,216_D01,L8TX_200206_01_A04,WMB-10Xv3-CB,cell,NaN,10Xv3,CB,Slc32a1-IRES-Cre;Ai14-507773,Slc32a1-IRES-Cre/wt;Ai14(RCL-tdT)/wt,...,#66493D,#32662E,#EEFF99,28,#CC0026,-0.734662,30.065327,-0.581734,-0.688576,-8.551579
ACAGAAATCGACGTCG-471_B04,ACAGAAATCGACGTCG,471_B04,L8TX_201217_01_B07,WMB-10Xv3-CB,cell,NaN,10Xv3,CB,Snap25-IRES2-Cre;Ai14-556108,Ai14(RCL-tdT)/wt,...,#66493D,#32662E,#EEFF99,28,#CC0026,-0.228925,22.244992,-0.276623,-0.688576,-7.861430
TTGCCTGCAGGAGGAG-216_C01,TTGCCTGCAGGAGGAG,216_C01,L8TX_200206_01_H03,WMB-10Xv3-CB,cell,NaN,10Xv3,CB,Slc32a1-IRES-Cre;Ai14-507773,Slc32a1-IRES-Cre/wt;Ai14(RCL-tdT)/wt,...,#66493D,#32662E,#EEFF99,28,#CC0026,-0.736562,36.717444,-1.290367,-0.688576,-9.094108
TCCATCGTCTCGCTTG-594_B02,TCCATCGTCTCGCTTG,594_B02,L8TX_210408_01_G10,WMB-10Xv3-CB,cell,NaN,10Xv3,CB,Snap25-IRES2-Cre;Ai14-570071,Snap25-IRES2-Cre/wt;Ai14(RCL-tdT)/wt,...,#66493D,#32662E,#EEFF99,28,#CC0026,2.167889,32.596541,1.802641,0.321969,-7.538399


## Gene Ontology

In [9]:
# For one module
from gprofiler import GProfiler

# Initialize GProfiler
gp = GProfiler(return_dataframe=True)

# Select genes from a specific module
module_name = "Immune-M1"
genes_of_module = modules_df[modules_df["module"] == module_name]["gene_name"].tolist()

# Perform GO enrichment analysis
go_results = gp.profile(organism='mmusculus', query=genes_of_module)

# Display the results
go_results.head()

,source,native,name,p_value,significant,description,term_size,query_size,intersection_size,effective_domain_size,precision,recall,query,parents
0,GO:BP,GO:0002376,immune system process,2.170563e-23,True,"""Any process involved in the development or fu...",2866,158,69,26963,0.436709,0.024075,query_1,[GO:0008150]
1,GO:BP,GO:0002682,regulation of immune system process,4.168139e-16,True,"""Any process that modulates the frequency, rat...",1590,158,45,26963,0.284810,0.028302,query_1,"[GO:0002376, GO:0050789]"
2,GO:BP,GO:0006955,immune response,2.255390e-15,True,"""Any immune system process that functions in t...",1991,158,49,26963,0.310127,0.024611,query_1,"[GO:0002376, GO:0050896]"
3,GO:CC,GO:0016020,membrane,6.622280e-14,True,"""A lipid bilayer along with all the proteins a...",10237,165,114,27195,0.690909,0.011136,query_1,[GO:0110165]
4,GO:BP,GO:0050865,regulation of cell activation,1.075343e-13,True,"""Any process that modulates the frequency, rat...",632,158,28,26963,0.177215,0.044304,query_1,"[GO:0001775, GO:0050794, GO:0051239]"


In [10]:
go_results

,source,native,name,p_value,significant,description,term_size,query_size,intersection_size,effective_domain_size,precision,recall,query,parents
0,GO:BP,GO:0002376,immune system process,2.170563e-23,True,"""Any process involved in the development or fu...",2866,158,69,26963,0.436709,0.024075,query_1,[GO:0008150]
1,GO:BP,GO:0002682,regulation of immune system process,4.168139e-16,True,"""Any process that modulates the frequency, rat...",1590,158,45,26963,0.284810,0.028302,query_1,"[GO:0002376, GO:0050789]"
2,GO:BP,GO:0006955,immune response,2.255390e-15,True,"""Any immune system process that functions in t...",1991,158,49,26963,0.310127,0.024611,query_1,"[GO:0002376, GO:0050896]"
3,GO:CC,GO:0016020,membrane,6.622280e-14,True,"""A lipid bilayer along with all the proteins a...",10237,165,114,27195,0.690909,0.011136,query_1,[GO:0110165]
4,GO:BP,GO:0050865,regulation of cell activation,1.075343e-13,True,"""Any process that modulates the frequency, rat...",632,158,28,26963,0.177215,0.044304,query_1,"[GO:0001775, GO:0050794, GO:0051239]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
264,GO:BP,GO:0002695,negative regulation of leukocyte activation,4.462604e-02,True,"""Any process that stops, prevents, or reduces ...",204,158,8,26963,0.050633,0.039216,query_1,"[GO:0002683, GO:0002694, GO:0045321, GO:0050866]"
265,TF,TF:M01288,Factor: NeuroD; motif: NNSCWGCTGNSY,4.681003e-02,True,Factor: NeuroD; motif: NNSCWGCTGNSY,4919,160,59,21628,0.368750,0.011994,query_1,[TF:M00000]
266,GO:BP,GO:0002831,regulation of response to biotic stimulus,4.751983e-02,True,"""Any process that modulates the frequency, rat...",565,158,13,26963,0.082278,0.023009,query_1,"[GO:0009607, GO:0048583]"
267,GO:BP,GO:2001280,positive regulation of unsaturated fatty acid ...,4.917327e-02,True,"""Any process that activates or increases the f...",11,158,3,26963,0.018987,0.272727,query_1,"[GO:0006636, GO:0045723, GO:2001279]"


In [11]:
# For all modules
from gprofiler import GProfiler

for module_name in modules:
    # Initialize GProfiler
    gp = GProfiler(return_dataframe=True)
    
    # Select genes from a specific module
    genes_of_module = modules_df[modules_df["module"] == module_name]["gene_name"].tolist()

    # Perform GO enrichment analysis
    go_results = gp.profile(organism='mmusculus', query=genes_of_module)

    # Display the results
    go_results.to_csv(f"/data/scRNA/ABCA/AIBS/AWS/expression_matrices/WMB-10Xv3/20230630/outputs/GO/{module_name}_GO.csv")
    print(f"Module saved: {module_name}")

Module saved: Immune-M1
Module saved: Immune-M2
Module saved: Immune-M3
Module saved: Immune-M4
Module saved: Immune-M5
